# Expression System — Variables and Symbolic Equations

**Part II · Geometric Algebra** — Tutorial 14

This tutorial introduces the `pytanga.expression` symbolic layer for composing GA
equations where only a few elements change (animation, optimization, fitting).
`Variable` is a named slot with a fixed blade mask; `Expression` is a reduced
product tensor built by combining variables with constants; `AffineExpression`
holds a sum of unmergeable terms.

By the end you will be able to:

- Declare `Variable`s and build `Expression`s with `*`, `|`, `^`, scalars, and
  repeated variables.
- Evaluate expressions for single values and batches.
- Read the Jacobian of a partially-evaluated expression.
- Merge terms with `+`/`-` and work with `AffineExpression`.
- Apply involutions (`~e`, `e.conj()`).
- Invert or least-squares-solve linear maps with `inv()`, `lstsq()`, and `svd()`.

> **Prerequisites:** [Tutorial 10](../10_blade_mask/) (blade masks) and
> [Tutorial 13](../13_tensor/) (tensors).

## 1. Setup

`Variable` is re-exported from the top-level package; `Expression` and
`AffineExpression` live in `pytanga.expression`. We bind one `BasisE3` algebra.

In [1]:
from pytanga import BladeMask, DataArray, Variable
from pytanga.basis import BasisE3

E3 = BasisE3()
full = BladeMask.full(E3)

## 2. `Variable` — a named slot

A `Variable` carries a name (used as the keyword when evaluating) and a fixed
`BladeMask` that restricts which blades it may hold. It carries no data itself.

In [2]:
v = Variable("V1", full)

print(repr(v))
print("name    :", v.name)
print("mask    :", v.mask.names())
print("algebra :", type(v.algebra).__name__)

Variable('V1', BladeMask(['s', 'e1', 'e2', 'e3', 'e12', 'e13', 'e23', 'I']))
name    : V1
mask    : ['s', 'e1', 'e2', 'e3', 'e12', 'e13', 'e23', 'I']
algebra : BasisE3


## 3. Building expressions

Combine variables with constants or other variables. Constant operands are folded
into the tensor at build time; each variable occurrence gets its own axis.

In [3]:
a = E3("2 e1")
b = E3("3 e2")
w = Variable("V2", full)

e_gp = v * a          # geometric product
e_ip = v | a          # inner product
e_op = v ^ a          # outer product
e_2v = v * w          # two-variable product
e_sc = 2.0 * v * a    # scalar scale
e_rep = v * v         # repeated variable (polynomial form)

print("type of v * a   :", type(e_gp).__name__)
print("shape of v * w  :", e_2v.tensor.shape)

type of v * a   : Expression
shape of v * w  : (8, 8, 8)


## 4. Evaluation — single and batched

Call an expression and bind variables by name. A single value returns an `MV`; a
`DataArray` binds a batch of values (a list of `MV`s) and returns a list of
`MV`s (one einsum), keeping the counting axis element-wise. See the DataArray
section below for counting-axis reduction.

In [4]:
x = E3("e1 + e2")

print("e_gp(x) :", e_gp(V1=x).prune().to_dict())
print("direct  :", (x * a).to_dict())

xs = [E3("e1"), E3("e2")]
print("batch   :", [m.to_dict() for m in e_gp(V1=DataArray(xs, masks=("n", full)))])

e_gp(x) : {'s': 2.0, 'e12': -2.0}
direct  : {'s': 2.0, 'e12': -2.0}
batch   : [{'s': 2.0}, {'e12': -2.0}]


## 5. Partial evaluation → Jacobians

Binding only some variables returns a new `Expression` over the remaining ones.
For a two-variable product this is the Jacobian of the map.

In [5]:
jac = e_2v(V1=E3("e1"))        # hold v = e1; still linear in w

print("type    :", type(jac).__name__)
print("shape   :", jac.tensor.shape)
print("jac(W2) :", jac(V2=E3("e2")).prune().to_dict())
print("direct  :", (E3("e1") * E3("e2")).to_dict())

type    : Expression
shape   : (8, 8)
jac(W2) : {'e12': 1.0}
direct  : {'e12': 1.0}


## 6. Addition, subtraction, and affine sums

`+`/`-` merges expressions that share the same variables in the same order. When
terms cannot be merged, the result is an `AffineExpression` — a sum of terms
evaluated term by term.

In [6]:
merged = v * a + v * b          # same variable -> single Expression
print("merged :", type(merged).__name__, "->", merged(V1=x).prune().to_dict())
print("direct :", (x * a + x * b).to_dict())

c = E3("e3")
f = (v * v) + v + c             # different shapes -> AffineExpression
print()
print("f      :", type(f).__name__, "with", len(f.terms), "terms")
print("f(x)   :", f(V1=x).to_dict())
print("direct :", ((x * x) + x + c).to_dict())

merged : Expression -> {'s': 5.0, 'e12': 1.0}
direct : {'s': 5.0, 'e12': 1.0}

f      : AffineExpression with 3 terms
f(x)   : {'s': 2.0, 'e1': 1.0, 'e2': 1.0, 'e3': 1.0}
direct : {'s': 2.0, 'e1': 1.0, 'e2': 1.0, 'e3': 1.0}


## 7. Involutions

`~e` (reverse) and `e.conj()` (Clifford conjugate) are diagonal sign tensors. For a
grade-2 blade, reverse negates the blade.

In [7]:
biv = BladeMask(E3, grades=[2])
V = Variable("V1", biv)
E = V * 1.0                    # identity on the bivector subspace

xb = E3("3 e12")
print("E(x)       :", E(V1=xb).prune().to_dict())
print("~E(x)      :", (~E)(V1=xb).prune().to_dict())
print("E.conj()(x):", E.conj()(V1=xb).prune().to_dict())

E(x)       : {'e12': 3.0}
~E(x)      : {'e12': -3.0}
E.conj()(x): {'e12': -3.0}


## 8. Inverse, least squares, and SVD

For a single-variable linear expression, `inv(name)` returns the inverse linear
map; `lstsq(rhs=...)` solves numerically; `svd()` returns singular values plus
right-singular vectors.

In [8]:
A = E3("1 + 2 e1 - e2 + 0.5 e3")
X_true = E3("3 - 2 e1 + e2 + 0.25 e3 + 1.5 I")
B = A * X_true

Xvar = Variable("X", full)
forward = A * Xvar                 # the linear map  X -> A * X

X_inv = forward.inv("X")(X=B)
print("inv recovered :", {k: round(v, 4) for k, v in X_inv.to_dict().items()})
print("|A*X - B|     :", (A * X_inv - B).mag)

inv recovered : {'s': 3.0, 'e1': -2.0, 'e2': 1.0, 'e12': 0.0, 'e3': 0.25, 'e13': 0.0, 'e23': 0.0, 'I': 1.5}
|A*X - B|     : 0.0


In [9]:
X_lstsq = forward.lstsq(rhs=B)
print("lstsq recovered:", {k: round(v, 4) for k, v in X_lstsq.to_dict().items()})

values, mvs = forward.svd()
print("svd singular values:", len(values))
print("smallest singular vector:", {k: round(v, 4) for k, v in mvs[-1].to_dict().items()})

lstsq recovered: {'s': 3.0, 'e1': -2.0, 'e2': 1.0, 'e12': 0.0, 'e3': 0.25, 'e13': -0.0, 'e23': 0.0, 'I': 1.5}
svd singular values: 8
smallest singular vector: {'s': -0.0, 'e1': 0.0327, 'e2': -0.1994, 'e12': 0.252, 'e3': -0.5296, 'e13': 0.6539, 'e23': 0.0943, 'I': -0.4227}


## 9. DataArray — batches and counting-axis reduction

`DataArray(array, masks=(...))` is the labeled data container of the expression
system. The `masks` are given **in the order of the array's dimensions**: a
`BladeMask` marks a blade axis, a `str` names a counting axis, and the blade axis
is not necessarily first or last. Pass it to bind a variable to a batch of values;
the counting axis is kept element-wise, and you can then **reduce** it:

- A raw 1-D array **sums** the axis away: `bound(axis=[w0, w1, ...])`.
- The `"_"` marker **keeps** it element-wise:
  `bound(axis=DataArray(w, masks=("_",)))`.
- The `"*"` marker is the explicit sum marker; for multi-axis data the other axes
  become new named dimensions. `rename_axis` / in-place `__call__` rename axes.


In [10]:
import numpy as np

# For a NumPy array, `masks` follow the array's dimension order; the blade axis
# is not necessarily first or last.
vec = BladeMask(E3, grades=[1])                            # e1, e2, e3
print(DataArray(np.random.rand(4, 3), masks=("n", vec)).masks)   # blade axis last
print(DataArray(np.random.rand(3, 4), masks=(vec, "n")).masks)   # blade axis first

xs = [E3("e1"), E3("e2"), E3("e3")]
batch = DataArray(xs, masks=("n", full))       # list of MVs: counting name + blade mask

bound = e_2v(V1=batch)                         # still over V2, with an "n" axis
print("bound  :", type(bound).__name__, bound.tensor.shape)

s = bound(n=[1.0, 2.0, 3.0])                   # sum the "n" axis away
print("summed :", s(V2=E3("e1")).prune().to_dict())

kept = bound(n=DataArray([1.0, 2.0, 3.0], masks=("_",)))   # keep element-wise
print("kept   :", [m.prune().to_dict() for m in kept(V2=E3("e1"))])


('n', BladeMask(['e1', 'e2', 'e3']))
(BladeMask(['e1', 'e2', 'e3']), 'n')
bound  : Expression (8, 8, 3)
summed : {'s': 1.0, 'e12': -2.0, 'e13': -3.0}
kept   : [{'s': 1.0}, {'e12': -2.0}, {'e13': -3.0}]


## 10. Worked example — weighted grid sum

Put the pieces together: build the weighted sum

$S = \sum_i \phi(x_i) * (x_i \wedge (x_i \cdot B))$

over the `10×10×10` points of a 3-D grid, where `B` is a bivector `Variable` and
`phi(x_i)` is a scalar weight computed from each point's distance to the grid
centre. Bind `x` to a `DataArray` of grid points (one blade axis + one counting
axis), then reduce that counting axis with the weight vector — the result is a
`3×3` linear map from the coefficients of `B`.


In [11]:
import numpy as np

point = BladeMask(E3, grades=[1])            # vectors
biv = BladeMask(E3, grades=[2])              # bivectors

B = Variable("B", biv)
x = Variable("x", point)

expr = x ^ (x | B)                           # x_i ^ (x_i | B)

grid = np.linspace(-1.0, 1.0, 10)
X, Y, Z = np.meshgrid(grid, grid, grid, indexing="ij")
points = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)   # (1000, 3)

r = np.linalg.norm(points, axis=1)
phi = np.exp(-0.5 * ((r - 0.3) / 0.4) ** 2) / 1000              # weights phi(x_i)

partial = expr(x=DataArray(points, masks=("pnt_idx", point)))   # still over B
S = partial(pnt_idx=phi)                                        # sum "pnt_idx" away

print("points  :", points.shape)
print("partial :", type(partial).__name__, partial.tensor.shape)
print("S       :", type(S).__name__, S.tensor.shape)


points  : (1000, 3)
partial : Expression (3, 3, 1000)
S       : Expression (3, 3)


In [12]:
B_val = E3("e12 + 2 e13 + 3 e23")
print("S(B)    :", S(B=B_val))


S(B)    : 0.1073 e12 + 0.2145 e13 + 0.3218 e23


## 11. Limits

- A variable may appear at most `MAX_DEGREE` (4) times per product term;
  exceeding this raises `ValueError`.
- `Variable` labels are **integers** from a monotonic, effectively unbounded
  pool, so there is no practical limit on the number of live variables (the old
  single-letter ceiling is gone). Counting-axis names in the underlying
  `MVLabeledTensor` may also be integers, via the `AxisLabel(name, mode)`
  dataclass (exported from `pytanga.tensor`), so axis names are not limited to
  the single-letter alphabet either.
- A single stacked expression composes with a constant or variable under `*`, and
  matching stacked expressions merge under `+`/`-`; two stacked operands cannot
  be composed with each other.


## 12. Summary & next steps

| Task | API |
|---|---|
| Named slot | `Variable("V1", mask)` |
| Build | `v * a`, `v ^ a`, `v * w`, `v * v` |
| Evaluate | `e(V1=x)`, `e(V1=DataArray(xs, masks=("n", full)))` |
| Constant | `Expression(A)`, `Expression(A, mask)` |
| Batch / reduce | `DataArray(array, masks=...)`, `e(axis=[...])` |
| Jacobian | `e(V1=x)` on a multi-variable expression |
| Merge / affine sum | `v*a + v*b`, `(v*v) + v + c` |
| Involutions | `~e`, `e.conj()` |
| Inverse / least squares / SVD | `e.inv(name)`, `e.lstsq(rhs=...)`, `e.svd()` |
| Underlying tensor | `e.tensor` |

**Where to go next:**

- [**15 · Geometry Submodule**](../15_geometry/) — algebra-independent entities and
  operators.
- [**11 · Equation Solving**](../11_equation_solving/) — the solver pipeline that
  expressions wrap symbolically.